In [ ]:
# ============================================================
# CELDA 1 - Conexión y carga de datos Csvs
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ruta = '/content/drive/MyDrive/Maestria/Diseño_Optimización_BD/Unidad1/BD/archive/'

customers     = pd.read_csv(ruta + 'olist_customers_dataset.csv')
orders        = pd.read_csv(ruta + 'olist_orders_dataset.csv')
order_items   = pd.read_csv(ruta + 'olist_order_items_dataset.csv')
order_payments= pd.read_csv(ruta + 'olist_order_payments_dataset.csv')
reviews       = pd.read_csv(ruta + 'olist_order_reviews_dataset.csv')
products      = pd.read_csv(ruta + 'olist_products_dataset.csv')
sellers       = pd.read_csv(ruta + 'olist_sellers_dataset.csv')
geolocation   = pd.read_csv(ruta + 'olist_geolocation_dataset.csv')
categories    = pd.read_csv(ruta + 'product_category_name_translation.csv')

tablas = {
    'customers': customers, 'orders': orders,
    'order_items': order_items, 'payments': order_payments,
    'reviews': reviews, 'products': products, 'sellers': sellers, 'geolocation':geolocation,'categories':categories
}
print("Datos cargados correctamente")

In [ ]:
# ============================================================
# CELDA 2 - EDA estructurado por cada tabla
# ============================================================
for nombre, df in tablas.items():
    print(f"\n{'='*50}")
    print(f" TABLA: {nombre.upper()}")
    print(f"   Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}")
    print(df.info())
    print("\n Estadísticas:")
    print(df.describe())
    print("\n Valores nulos:")
    nulos = df.isnull().sum()
    print(nulos[nulos > 0] if nulos.sum() > 0 else "   Sin nulos")

In [ ]:
# ============================================================
# CELDA 3 - Visualización temporal
# ============================================================
orders['order_purchase_timestamp'] = pd.to_datetime(
    orders['order_purchase_timestamp'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pedidos por mes
orders.groupby(orders['order_purchase_timestamp'].dt.month) \
    .size().plot(kind='bar', ax=axes[0], color='green')
axes[0].set_title('Pedidos por Mes')
axes[0].set_xlabel('Mes')
axes[0].set_ylabel('Cantidad')

# Pedidos por año
orders.groupby(orders['order_purchase_timestamp'].dt.year) \
    .size().plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Pedidos por Año')
axes[1].set_xlabel('Año')

plt.tight_layout()
plt.savefig('distribucion_temporal.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# CELDA 4 - Distribución geográfica
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Top 10 estados con más clientes
customers['customer_state'].value_counts().head(10) \
    .plot(kind='bar', ax=axes[0], color='teal')
axes[0].set_title('Top 10 Estados - Clientes')
axes[0].set_xlabel('Estado')
axes[0].set_ylabel('Cantidad')

# Top 10 estados con más vendedores
sellers['seller_state'].value_counts().head(10) \
    .plot(kind='bar', ax=axes[1], color='purple')
axes[1].set_title('Top 10 Estados - Vendedores')
axes[1].set_xlabel('Estado')

plt.tight_layout()
plt.savefig('distribucion_geografica.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# CELDA 5 - Análisis de nulos por tabla
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

nulos_data = {
    nombre: df.isnull().sum()
    for nombre, df in tablas.items()
    if df.isnull().sum().sum() > 0
}

nulos_items = list(nulos_data.items())[:4]

for i, ax in enumerate(axes.flatten()):
    if i < len(nulos_items):
        nombre, nulos = nulos_items[i]
        nulos_con_valor = nulos[nulos > 0]
        nulos_con_valor.plot(kind='bar', ax=ax, color='tomato')
        ax.set_title(f'Nulos en {nombre}')
        ax.set_ylabel('Cantidad')
        ax.tick_params(axis='x', rotation=45)
    else:
        ax.axis('off')

plt.tight_layout()
plt.savefig('nulos_por_tabla.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# CELDA 6 - Relaciones entre tablas
# ============================================================
print("RELACIONES ENTRE TABLAS")
print("="*50)

# Orders + Customers
oc = orders.merge(customers, on='customer_id', how='left')
print(f"orders + customers: {oc.shape}")

# Orders + Items
oi = orders.merge(order_items, on='order_id', how='left')
print(f"orders + order_items: {oi.shape}")

# Items + Products
ip = order_items.merge(products, on='product_id', how='left')
print(f"order_items + products: {ip.shape}")

# Orders + Payments
op = orders.merge(order_payments, on='order_id', how='left')
print(f"orders + payments: {op.shape}")

# Orders + Reviews
orv = orders.merge(reviews, on='order_id', how='left')
print(f"orders + reviews: {orv.shape}")

print("\nTodas las relaciones validadas por claves foráneas")